# Score ablation — validity holds for any score

A second experiment that **reuses the frozen `graded_claims.jsonl`** from the main
run (same claims, labels, tiers) and swaps in four different scores:

| | score | mechanism |
|---|---|---|
| S1 | **P(True)** | the generator's own true/false self-judgment *(already in `confidence`)* |
| S2 | **SeqLogProb** | length-normalized log-prob of the claim — generation *likelihood* |
| S4 | **NLI-entail** | DeBERTa-MNLI entailment of the claim by the answer — a *separate model* |
| -- | **Random** | pure noise — the floor |

The point: severity-aware CRC keeps the **dangerous risk <= 0.05 for every one of
them** (validity is distribution-free), while **retention tracks score quality**.

### Before running
- Settings -> **GPU** on, **Internet** on.
- **+ Add Input**: the code dataset with the `sac/` folder -> set `SAC_PATH`.
- **+ Add Input**: the dataset holding your `graded_claims.jsonl` (the main-run output).
- Add-ons -> **Secrets**: `HF_TOKEN` (gated Llama-3).
- Then **Run All**. S2/S4 checkpoint per claim, so a dead session just resumes.

## Config + setup

In [ ]:
# ===================== CONFIG — edit, then Run All =====================
SAC_PATH        = "/kaggle/input/severity-aware-conformal"   # folder with the sac/ package
GEN_MODEL       = "meta-llama/Meta-Llama-3-8B-Instruct"      # same generator as the main run
NLI_MODEL       = "microsoft/deberta-large-mnli"             # separate entailment model (S4)
ALPHA_MARGINAL  = 0.10
ALPHA_DANGEROUS = 0.05
ALPHA_BENIGN    = 0.15
N_SPLITS        = 300
RANDOM_SEED     = 0
WORK            = "/kaggle/working"
# ======================================================================

In [ ]:
!pip -q install -U transformers bitsandbytes accelerate

import os, sys, glob, shutil, json
sys.path.insert(0, SAC_PATH)

KQA     = f"{WORK}/kqa.jsonl"
GRADED  = f"{WORK}/graded_claims.jsonl"     # frozen main-run output (INPUT to this notebook)
ANSWERS = f"{WORK}/answers.jsonl"           # regenerated greedy answers (NLI premises)
SCORES  = f"{WORK}/ablation_scores.jsonl"   # per-claim S2 + S4 scores

!wget -q -O {KQA} https://raw.githubusercontent.com/Itaymanes/K-QA/main/dataset/questions_w_answers.jsonl

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))   # Llama-3 is gated

import numpy as np
from tqdm.auto import tqdm
from sac.kqa_loader import load_kqa
from sac.cache import load_claims
from sac.scores import random_scores, swap_confidence
from sac.ablation import run_ablation

# ---- restore the frozen graded claims + any partial ablation caches from inputs ----
for cache in (GRADED, ANSWERS, SCORES):
    if not os.path.exists(cache):
        hits = glob.glob(f"/kaggle/input/**/{os.path.basename(cache)}", recursive=True)
        if hits:
            shutil.copy(hits[0], cache); print("restored", os.path.basename(cache))

claims    = [c for c in load_claims(GRADED) if c.label in (0, 1)]   # verifiable only
questions = {it.qid: it.question for it in load_kqa(KQA)}
print(len(claims), "verifiable claims across", len({c.answer_id for c in claims}), "questions")

## Part A — regenerate greedy answers (GPU, checkpointed -> `answers.jsonl`)

NLI needs the *source answer* as premise. Greedy decoding is deterministic, so this
reproduces the same answers the claims were decomposed from. ~201 generations.

In [ ]:
from sac.hf_backend import HFBackend

def _load_answers(path):
    if not os.path.exists(path):
        return {}
    return {json.loads(l)["qid"]: json.loads(l)["answer"] for l in open(path)}

answers = _load_answers(ANSWERS)
gen     = HFBackend(model_id=GEN_MODEL)
need    = [qid for qid in {c.answer_id for c in claims} if qid not in answers]
print(len(answers), "answers cached,", len(need), "to generate")
for qid in tqdm(need, desc="answers"):
    ans = gen.generate(f"Question: {questions[qid]}\nAnswer:")
    answers[qid] = ans
    with open(ANSWERS, "a") as f:
        f.write(json.dumps({"qid": qid, "answer": ans}) + "\n")
print("answers ready:", len(answers))

## Part B — compute S2 (SeqLogProb) and S4 (NLI) per claim (GPU, checkpointed -> `ablation_scores.jsonl`)

P(True) is already each claim's `confidence`; Random is generated on the fly. Only
these two need the GPU. One line per claim, so a dead session resumes where it left off.

In [ ]:
from sac.nli_backend import NLIBackend

done = {json.loads(l)["claim_id"] for l in open(SCORES)} if os.path.exists(SCORES) else set()
pending = [c for c in claims if c.claim_id not in done]
print(len(done), "claims scored,", len(pending), "pending")

nli = NLIBackend(model_id=NLI_MODEL)   # loads alongside Llama-4bit; both fit on a T4
for c in tqdm(pending, desc="S2+S4"):
    ctx = f"Question: {questions[c.answer_id]}\nClaim: "
    s2  = gen.seq_logprob(ctx, c.text)                    # S2: generation likelihood
    s4  = nli.entail_prob(answers[c.answer_id], c.text)   # S4: entailment vs source answer
    with open(SCORES, "a") as f:
        f.write(json.dumps({"claim_id": c.claim_id, "seqlogprob": s2, "nli": s4}) + "\n")
print("scoring done:", len(claims), "claims")

## Part C — run the ablation (CPU)

Swap each score into the *same* claims and rerun the global-vs-severity-aware
headline over `N_SPLITS` splits. Read the table top to bottom: AUROC falls,
retention falls with it, but `dRisk-S` (severity-aware dangerous risk) never
breaks the 0.05 budget.

In [ ]:
alt   = {json.loads(l)["claim_id"]: json.loads(l) for l in open(SCORES)}
seqlp = {cid: v["seqlogprob"] for cid, v in alt.items()}
nli_s = {cid: v["nli"]        for cid, v in alt.items()}
rnd   = random_scores([c.claim_id for c in claims], seed=RANDOM_SEED)

scorers = {
    "P(True) [S1]":    claims,                       # existing confidence
    "SeqLogProb [S2]": swap_confidence(claims, seqlp),
    "NLI-entail [S4]": swap_confidence(claims, nli_s),
    "Random":          swap_confidence(claims, rnd),
}

print(f"Score ablation — averaged over {N_SPLITS} splits  "
      f"(alpha: marg={ALPHA_MARGINAL} danger={ALPHA_DANGEROUS} benign={ALPHA_BENIGN})\n")
hdr = (f"{'score':16s} {'AUROC':>6s} | {'dRisk-G':>7s} {'dRisk-S':>7s} | "
       f"{'dRet-G':>6s} {'dRet-S':>6s} | {'P>.05-G':>7s} {'P>.05-S':>7s}")
print(hdr); print("-" * len(hdr))
for name, cl in scorers.items():
    r = run_ablation(cl, ALPHA_MARGINAL, ALPHA_DANGEROUS, ALPHA_BENIGN, N_SPLITS)
    print(f"{name:16s} {r['auroc']:6.3f} | {r['g_d_risk']:7.3f} {r['s_d_risk']:7.3f} | "
          f"{r['g_d_ret']:6.3f} {r['s_d_ret']:6.3f} | {r['g_violation']:7.2f} {r['s_violation']:7.2f}")
print("\nValidity: severity-aware dRisk-S stays <= 0.05 for EVERY score (even Random).")
print("Quality : severity-aware retention dRet-S tracks AUROC — better scores keep more truth.")